In [ ]:
import os
import shutil
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
import ast

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/twincar")

print("Project exists:", PROJECT_DIR.exists())
print("Project path:", PROJECT_DIR)

print("\nImportant files:")
for path in [
    PROJECT_DIR / "models" / "convnext_tiny" / "best_model.pt",
    PROJECT_DIR / "models" / "convnext_tiny" / "idx_to_class.json",
    PROJECT_DIR / "models" / "convnext_tiny" / "train_config.json",
]:
    print(path, "->", path.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project exists: True
Project path: /content/drive/MyDrive/twincar

Important files:
/content/drive/MyDrive/twincar/models/convnext_tiny/best_model.pt -> True
/content/drive/MyDrive/twincar/models/convnext_tiny/idx_to_class.json -> True
/content/drive/MyDrive/twincar/models/convnext_tiny/train_config.json -> True


In [ ]:
SCRIPTS_DIR = PROJECT_DIR / "scripts"
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print("Scripts folder:", SCRIPTS_DIR)
print("Exists:", SCRIPTS_DIR.exists())

Scripts folder: /content/drive/MyDrive/twincar/scripts
Exists: True


In [ ]:
SAMPLE_DIR = Path("/content/drive/MyDrive/twincar/sample_images")
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

print("Put test/demo car images here:")
print(SAMPLE_DIR)

Put test/demo car images here:
/content/drive/MyDrive/twincar/sample_images


In [ ]:
TEST_DIR = Path("/content/twincar_data/test")

print("Test dir exists:", TEST_DIR.exists())

if TEST_DIR.exists():
    test_images = list(TEST_DIR.rglob("*.jpg"))
    print("Number of test images:", len(test_images))
    print("Example image:", test_images[0] if len(test_images) > 0 else "No images")

Test dir exists: False


In [ ]:
DATA_DIR = Path("/content/twincar_data")

STANFORD_CACHE = PROJECT_DIR / "stanford_cars_cache"

METADATA_PATH = PROJECT_DIR / "data" / "stanford_metadata.csv"

TEST_DIR = DATA_DIR / "test"

print("Project dir:", PROJECT_DIR)
print("Metadata exists:", METADATA_PATH.exists())
print("Test dir:", TEST_DIR)

Project dir: /content/drive/MyDrive/twincar
Metadata exists: True
Test dir: /content/twincar_data/test


In [ ]:
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def sanitize(name):
    """
    Converts class name into folder name.

    Example:
    'Audi S4 Sedan 2012' -> 'Audi_S4_Sedan_2012'
    """
    return str(name).replace(" ", "_").replace("/", "-")


def get_class_name(row):
    """
    Use class_name if it exists in metadata.
    Otherwise fallback to car_name.
    """
    if "class_name" in row and pd.notna(row["class_name"]):
        return row["class_name"]
    return row["car_name"]


def count_images_in_folder(path: Path):
    if not path.exists():
        return 0

    return sum(
        1
        for f in path.rglob("*")
        if f.is_file() and f.suffix.lower() in IMG_EXTENSIONS
    )


def count_test_images():
    return count_images_in_folder(TEST_DIR)


if count_test_images() == 0:
    print("Test images missing. Restoring Stanford Cars test split...")

    if not METADATA_PATH.exists():
        raise FileNotFoundError(
            f"Missing metadata file: {METADATA_PATH}\n"
            "Run 02_data_preparation.ipynb first."
        )

    dataset = load_dataset(
        "naufalso/stanford_cars",
        cache_dir=str(STANFORD_CACHE)
    )

    metadata = pd.read_csv(METADATA_PATH)
    test_df = metadata[metadata["split"] == "test"].reset_index(drop=True)

    if len(test_df) == 0:
        raise ValueError("No rows with split == 'test' found in metadata.")

    if TEST_DIR.exists():
        shutil.rmtree(TEST_DIR)

    TEST_DIR.mkdir(parents=True, exist_ok=True)

    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Restoring test images"):
        class_name = get_class_name(row)
        class_dir = TEST_DIR / sanitize(class_name)
        class_dir.mkdir(parents=True, exist_ok=True)

        hf_idx = int(row["hf_idx"])
        img = dataset["test"][hf_idx]["image_path"]

        img.save(class_dir / f"{hf_idx}.jpg")

    del dataset

    print(f"Copied test images: {len(test_df)}")

else:
    print("Test images already exist. Skipping restore.")

Test images missing. Restoring Stanford Cars test split...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/614 [00:00<?, ?B/s]

Restoring test images:   0%|          | 0/8000 [00:00<?, ?it/s]

Copied test images: 8000


In [ ]:
assert TEST_DIR.exists(), f"Test directory does not exist: {TEST_DIR}"

class_folders = [p for p in TEST_DIR.iterdir() if p.is_dir()]
num_images = count_test_images()

print("Test directory:", TEST_DIR)
print("Number of test images:", num_images)
print("Number of class folders:", len(class_folders))
print("Example class folders:")
print([p.name for p in class_folders[:10]])

Test directory: /content/twincar_data/test
Number of test images: 8000
Number of class folders: 195
Example class folders:
['Suzuki_Aerio_Sedan_2007', 'Buick_Verano_Sedan_2012', 'Dodge_Charger_SRT-8_2009', 'Dodge_Ram_Pickup_3500_Quad_Cab_2009', 'Infiniti_QX56_SUV_2011', 'GMC_Yukon_Hybrid_SUV_2012', 'Jeep_Grand_Cherokee_SUV_2012', 'BMW_6_Series_Convertible_2007', 'Chevrolet_Express_Van_2007', 'Ferrari_458_Italia_Convertible_2012']


In [ ]:
!python /content/drive/MyDrive/twincar/scripts/batch_predict.py \
  --image_dir /content/twincar_data/test \
  --recursive \
  --model_path /content/drive/MyDrive/twincar/models/convnext_tiny_updated_preprocess/best_model.pt \
  --label_map /content/drive/MyDrive/twincar/models/convnext_tiny_updated_preprocess/idx_to_class.json \
  --config /content/drive/MyDrive/twincar/models/convnext_tiny_updated_preprocess/train_config.json \
  --output /content/drive/MyDrive/twincar/reports/batch_predictions_test.csv \
  --model_name convnext_tiny \
  --top_k 5

TwinCar batch prediction
Device       : cuda
Model name   : convnext_tiny
Model path   : /content/drive/MyDrive/twincar/models/convnext_tiny_updated_preprocess/best_model.pt
Label map    : /content/drive/MyDrive/twincar/models/convnext_tiny_updated_preprocess/idx_to_class.json
Classes      : 195
Images found : 8000
Saved predictions to: /content/drive/MyDrive/twincar/reports/batch_predictions_test.csv

                                                    image_path  predicted_idx             predicted_class predicted_make predicted_model   predicted_make_model predicted_year  confidence                                                                                                                                                                        top_k_predictions                                top_k_probabilities
/content/twincar_data/test/AM_General_Hummer_SUV_2000/1107.jpg            124 HUMMER H2 SUT Crew Cab 2009         HUMMER H2 SUT Crew Cab HUMMER H2 SUT Crew Cab           2

In [ ]:
PRED_CSV = Path("/content/drive/MyDrive/twincar/reports/batch_predictions_test.csv")

df = pd.read_csv(PRED_CSV)

print("Rows:", len(df))
df.head()

Rows: 8000


,image_path,predicted_idx,predicted_class,predicted_make,predicted_model,predicted_make_model,predicted_year,confidence,top_k_predictions,top_k_probabilities
0,/content/twincar_data/test/AM_General_Hummer_S...,124,HUMMER H2 SUT Crew Cab 2009,HUMMER,H2 SUT Crew Cab,HUMMER H2 SUT Crew Cab,2009,0.574187,"[""HUMMER H2 SUT Crew Cab 2009"", ""HUMMER H3T Cr...","[0.574187, 0.108103, 0.101463, 0.01232, 0.004622]"
1,/content/twincar_data/test/AM_General_Hummer_S...,0,AM General Hummer SUV 2000,AM General,Hummer SUV,AM General Hummer SUV,2000,0.912600,"[""AM General Hummer SUV 2000"", ""HUMMER H2 SUT ...","[0.9126, 0.004012, 0.003252, 0.001123, 0.000911]"
2,/content/twincar_data/test/AM_General_Hummer_S...,0,AM General Hummer SUV 2000,AM General,Hummer SUV,AM General Hummer SUV,2000,0.913258,"[""AM General Hummer SUV 2000"", ""HUMMER H2 SUT ...","[0.913258, 0.002286, 0.001542, 0.001222, 0.001..."
3,/content/twincar_data/test/AM_General_Hummer_S...,0,AM General Hummer SUV 2000,AM General,Hummer SUV,AM General Hummer SUV,2000,0.888845,"[""AM General Hummer SUV 2000"", ""Dodge Sprinter...","[0.888845, 0.002662, 0.002395, 0.00157, 0.001415]"
4,/content/twincar_data/test/AM_General_Hummer_S...,0,AM General Hummer SUV 2000,AM General,Hummer SUV,AM General Hummer SUV,2000,0.924323,"[""AM General Hummer SUV 2000"", ""Ford F-150 Reg...","[0.924323, 0.001286, 0.001059, 0.001004, 0.000..."


In [ ]:
def folder_to_class_name(path):
    """
    Converts folder name back to Stanford Cars class name.

    Example:
    AM_General_Hummer_SUV_2000 -> AM General Hummer SUV 2000
    """
    folder_name = Path(path).parent.name
    return folder_name.replace("_", " ")


df["true_class"] = df["image_path"].apply(folder_to_class_name)

df[["image_path", "true_class", "predicted_class", "confidence"]].head(10)

,image_path,true_class,predicted_class,confidence
0,/content/twincar_data/test/AM_General_Hummer_S...,AM General Hummer SUV 2000,HUMMER H2 SUT Crew Cab 2009,0.574187
1,/content/twincar_data/test/AM_General_Hummer_S...,AM General Hummer SUV 2000,AM General Hummer SUV 2000,0.912600
2,/content/twincar_data/test/AM_General_Hummer_S...,AM General Hummer SUV 2000,AM General Hummer SUV 2000,0.913258
3,/content/twincar_data/test/AM_General_Hummer_S...,AM General Hummer SUV 2000,AM General Hummer SUV 2000,0.888845
4,/content/twincar_data/test/AM_General_Hummer_S...,AM General Hummer SUV 2000,AM General Hummer SUV 2000,0.924323
5,/content/twincar_data/test/AM_General_Hummer_S...,AM General Hummer SUV 2000,AM General Hummer SUV 2000,0.850840
6,/content/twincar_data/test/AM_General_Hummer_S...,AM General Hummer SUV 2000,AM General Hummer SUV 2000,0.922460
7,/content/twincar_data/test/AM_General_Hummer_S...,AM General Hummer SUV 2000,AM General Hummer SUV 2000,0.885588
8,/content/twincar_data/test/AM_General_Hummer_S...,AM General Hummer SUV 2000,AM General Hummer SUV 2000,0.899459
9,/content/twincar_data/test/AM_General_Hummer_S...,AM General Hummer SUV 2000,AM General Hummer SUV 2000,0.881971


In [ ]:
top1_accuracy = (df["true_class"] == df["predicted_class"]).mean()

print(f"Top-1 accuracy: {top1_accuracy:.4f}")
print(f"Top-1 accuracy: {top1_accuracy * 100:.2f}%")

Top-1 accuracy: 0.8836
Top-1 accuracy: 88.36%


Number of wrong predictions: 931
Number of correct predictions: 7069


,image_path,true_class,predicted_class,confidence,top_k_predictions
0,/content/twincar_data/test/AM_General_Hummer_S...,AM General Hummer SUV 2000,HUMMER H2 SUT Crew Cab 2009,0.574187,"[""HUMMER H2 SUT Crew Cab 2009"", ""HUMMER H3T Cr..."
20,/content/twincar_data/test/AM_General_Hummer_S...,AM General Hummer SUV 2000,HUMMER H2 SUT Crew Cab 2009,0.837812,"[""HUMMER H2 SUT Crew Cab 2009"", ""AM General Hu..."
31,/content/twincar_data/test/AM_General_Hummer_S...,AM General Hummer SUV 2000,Jeep Wrangler SUV 2012,0.409147,"[""Jeep Wrangler SUV 2012"", ""AM General Hummer ..."
32,/content/twincar_data/test/AM_General_Hummer_S...,AM General Hummer SUV 2000,HUMMER H2 SUT Crew Cab 2009,0.771609,"[""HUMMER H2 SUT Crew Cab 2009"", ""AM General Hu..."
34,/content/twincar_data/test/AM_General_Hummer_S...,AM General Hummer SUV 2000,Nissan NV Passenger Van 2012,0.359973,"[""Nissan NV Passenger Van 2012"", ""smart fortwo..."
51,/content/twincar_data/test/Acura_Integra_Type_...,Acura Integra Type R 2001,Lamborghini Diablo Coupe 2001,0.123569,"[""Lamborghini Diablo Coupe 2001"", ""Acura Integ..."
75,/content/twincar_data/test/Acura_Integra_Type_...,Acura Integra Type R 2001,Nissan 240SX Coupe 1998,0.192471,"[""Nissan 240SX Coupe 1998"", ""Audi 100 Sedan 19..."
88,/content/twincar_data/test/Acura_RL_Sedan_2012...,Acura RL Sedan 2012,Honda Accord Sedan 2012,0.398542,"[""Honda Accord Sedan 2012"", ""Acura RL Sedan 20..."
97,/content/twincar_data/test/Acura_RL_Sedan_2012...,Acura RL Sedan 2012,Acura TL Sedan 2012,0.923904,"[""Acura TL Sedan 2012"", ""Acura RL Sedan 2012"",..."
98,/content/twincar_data/test/Acura_RL_Sedan_2012...,Acura RL Sedan 2012,Suzuki SX4 Sedan 2012,0.108390,"[""Suzuki SX4 Sedan 2012"", ""Acura RL Sedan 2012..."


In [ ]:
confused_pairs = (
    wrong_df
    .groupby(["true_class", "predicted_class"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

confused_pairs.head(20)

,true_class,predicted_class,count
351,GMC Savana Van 2012,Chevrolet Express Van 2007,21
120,BMW 6 Series Convertible 2007,BMW M6 Convertible 2010,16
247,Chevrolet Tahoe Hybrid SUV 2012,Chevrolet Avalanche Crew Cab 2012,15
206,Chevrolet Express Van 2007,Chevrolet Express Cargo Van 2007,14
242,Chevrolet Silverado 2500HD Regular Cab 2012,Chevrolet Silverado 1500 Regular Cab 2012,12
101,Audi TTS Coupe 2012,Audi TT Hatchback 2011,12
203,Chevrolet Express Cargo Van 2007,Chevrolet Express Van 2007,11
83,Audi S5 Coupe 2012,Audi A5 Coupe 2012,11
274,Dodge Caliber Wagon 2012,Dodge Caliber Wagon 2007,10
104,Audi V8 Sedan 1994,Audi 100 Sedan 1994,9


In [ ]:
CONFUSED_PAIRS_PATH = "/content/drive/MyDrive/twincar/reports/batch_confused_pairs_test.csv"

confused_pairs.to_csv(CONFUSED_PAIRS_PATH, index=False)

print("Saved to:", CONFUSED_PAIRS_PATH)

Saved to: /content/drive/MyDrive/twincar/reports/batch_confused_pairs_test.csv


In [ ]:
MODELS_DIR = PROJECT_DIR / "models"
DATA_DIR = PROJECT_DIR / "data"

search_names = {
    "best_model.pt",
    "last_checkpoint.pt",
    "train_config.json",
    "idx_to_class.json",
    "class_to_idx.json",
    "imagefolder_label_map.json",
}

print("Searching model/data folders...\n")

for base_dir in [MODELS_DIR, DATA_DIR]:
    print(f"Inside: {base_dir}")
    for p in sorted(base_dir.rglob("*")):
        if p.name in search_names:
            size_mb = p.stat().st_size / 1e6
            print(f"{size_mb:8.2f} MB  |  {p}")
    print()

Searching model/data folders...

Inside: /content/drive/MyDrive/twincar/models
   17.33 MB  |  /content/drive/MyDrive/twincar/models/best_model.pt
  111.95 MB  |  /content/drive/MyDrive/twincar/models/convnext_tiny/best_model.pt
    0.01 MB  |  /content/drive/MyDrive/twincar/models/convnext_tiny/idx_to_class.json
  335.87 MB  |  /content/drive/MyDrive/twincar/models/convnext_tiny/last_checkpoint.pt
    0.00 MB  |  /content/drive/MyDrive/twincar/models/convnext_tiny/train_config.json
    0.05 MB  |  /content/drive/MyDrive/twincar/models/convnext_tiny_compcars_make_model_year/class_to_idx.json
    0.05 MB  |  /content/drive/MyDrive/twincar/models/convnext_tiny_compcars_make_model_year/idx_to_class.json
    0.00 MB  |  /content/drive/MyDrive/twincar/models/convnext_tiny_compcars_make_model_year/train_config.json
  111.95 MB  |  /content/drive/MyDrive/twincar/models/convnext_tiny_updated_preprocess/best_model.pt
    0.01 MB  |  /content/drive/MyDrive/twincar/models/convnext_tiny_updated_pr

In [ ]:
!python /content/drive/MyDrive/twincar/scripts/batch_predict.py \
  --image_dir /content/twincar_data/test \
  --recursive \
  --model_path /content/drive/MyDrive/twincar/models/best_model.pt \
  --label_map /content/drive/MyDrive/twincar/data/imagefolder_label_map.json \
  --config /content/drive/MyDrive/twincar/models/train_config.json \
  --output /content/drive/MyDrive/twincar/reports/batch_predictions_efficientnet_b0_v1_test.csv \
  --model_name efficientnet_b0 \
  --batch_size 32 \
  --top_k 5

TwinCar batch prediction
Device       : cuda
Model name   : efficientnet_b0
Model path   : /content/drive/MyDrive/twincar/models/best_model.pt
Label map    : /content/drive/MyDrive/twincar/data/imagefolder_label_map.json
Classes      : 195
Images found : 8000
Saved predictions to: /content/drive/MyDrive/twincar/reports/batch_predictions_efficientnet_b0_v1_test.csv

                                                    image_path  predicted_idx            predicted_class predicted_make predicted_model  predicted_make_model predicted_year  confidence                                                                                                                                                                      top_k_predictions                                top_k_probabilities
/content/twincar_data/test/AM_General_Hummer_SUV_2000/1107.jpg              0 AM General Hummer SUV 2000     AM General      Hummer SUV AM General Hummer SUV           2000    0.347307                             

In [ ]:
!python /content/drive/MyDrive/twincar/scripts/batch_predict.py \
  --image_dir /content/twincar_data/test \
  --recursive \
  --model_path /content/drive/MyDrive/twincar/models/efficientnet_b0_v2/best_model.pt \
  --label_map /content/drive/MyDrive/twincar/models/efficientnet_b0_v2/idx_to_class.json \
  --config /content/drive/MyDrive/twincar/models/efficientnet_b0_v2/train_config.json \
  --output /content/drive/MyDrive/twincar/reports/batch_predictions_efficientnet_b0_v2_test.csv \
  --model_name efficientnet_b0 \
  --batch_size 32 \
  --top_k 5

TwinCar batch prediction
Device       : cuda
Model name   : efficientnet_b0
Model path   : /content/drive/MyDrive/twincar/models/efficientnet_b0_v2/best_model.pt
Label map    : /content/drive/MyDrive/twincar/models/efficientnet_b0_v2/idx_to_class.json
Classes      : 195
Images found : 8000
Saved predictions to: /content/drive/MyDrive/twincar/reports/batch_predictions_efficientnet_b0_v2_test.csv

                                                    image_path  predicted_idx             predicted_class predicted_make predicted_model   predicted_make_model predicted_year  confidence                                                                                                                                                       top_k_predictions                                top_k_probabilities
/content/twincar_data/test/AM_General_Hummer_SUV_2000/1107.jpg            123 HUMMER H2 SUT Crew Cab 2009         HUMMER H2 SUT Crew Cab HUMMER H2 SUT Crew Cab           2009    0.397303         